In [40]:
from skimage import io
import sklearn
import numpy as np
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import skimage.transform as transform
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score

In [55]:
# Importing datasets containing womens clothing and mens clothing
image_size = (100, 100, 3)
women = io.imread_collection('/Users/Colette/anaconda/1CS156/Week_5_PCA/women/*')
men = io.imread_collection('/Users/Colette/anaconda/1CS156/Week_5_PCA/men/*')

# Resizing all of the images so they are uniform size
resized_women = []
for image in women:
    resized_women.append(transform.resize(image, image_size))
    
resized_men = []
for image in men:
    resized_men.append(transform.resize(image, image_size))

n_men = len(resized_men)

total_resized = np.concatenate((np.array(resized_women), np.array(resized_men)))
del(resized_women)
del(resized_men)

In [56]:
# Center all images around 0
reg_images = total_resized - total_resized.mean(axis=0)
reg_images.shape

# Reshaping to be a two dimensional array (two_dimension_array hehe)
two_dimension_array = np.reshape(reg_images, (402, 3 * 100 * 100))
two_dimension_array.shape

(402, 30000)

In [57]:
# Split the training set and test set for LogReg on the original dataset''
classes = [1] * n_women + [0] * n_men
X_train, X_test, y_train, y_test = train_test_split(two_dimension_array, classes, test_size=0.2, stratify=classes, random_state = 7) 

In [58]:
# Creating a Logistic Regression model to use for classification
from sklearn.linear_model import LogisticRegression

LogReg = LogisticRegression()
LogReg.fit(X_train, y_train)

LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
          intercept_scaling=1, max_iter=100, multi_class='ovr', n_jobs=1,
          penalty='l2', random_state=None, solver='liblinear', tol=0.0001,
          verbose=0, warm_start=False)

In [59]:
# Using the confusion matrix and accuracy score 
predictions_train = LogReg.predict(X_train)
score_train = LogReg.score(X_train, y_train)
print metrics.confusion_matrix(y_train, predictions_train)
print metrics.classification_report(y_train, predictions_train)
print metrics.accuracy_score(y_train, predictions_train)

predictions_test = LogReg.predict(X_test)
score_train = LogReg.score(X_test, y_test)
print metrics.confusion_matrix(y_test, predictions_test)
print metrics.classification_report(y_test, predictions_test)
print metrics.accuracy_score(y_test, predictions_test)

# model is super overfitting (look how poorly it does on test set)
# curse of dimensionality (so many dimensions)
# is overfitting on every single metric (number of variables is so much higher than number of images)

[[161   0]
 [  0 160]]
             precision    recall  f1-score   support

          0       1.00      1.00      1.00       161
          1       1.00      1.00      1.00       160

avg / total       1.00      1.00      1.00       321

1.0
[[32  8]
 [10 31]]
             precision    recall  f1-score   support

          0       0.76      0.80      0.78        40
          1       0.79      0.76      0.77        41

avg / total       0.78      0.78      0.78        81

0.777777777778


PCA

In [46]:
# PCA with a random number of components, seeing as though time is running out!
pca = PCA(n_components = 10)
pca.fit(two_dimension_array)
pca_array = pca.transform(two_dimension_array)

In [47]:
# Split the training set and test set for LogReg on the PCA dataset
classes = [1] * n_women + [0] * n_men
PCA_X_train, PCA_X_test, PCA_y_train, PCA_y_test = train_test_split(pca_array, classes, test_size=0.2, stratify=classes, random_state = 7) 

In [48]:
# Building a LogReg model from the split of the PCA data
LogReg = LogisticRegression()
LogReg.fit(PCA_X_train, PCA_y_train)

LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
          intercept_scaling=1, max_iter=100, multi_class='ovr', n_jobs=1,
          penalty='l2', random_state=None, solver='liblinear', tol=0.0001,
          verbose=0, warm_start=False)

In [49]:
# ModelMetrics
predictions_train = LogReg.predict(PCA_X_train)
score_train = LogReg.score(PCA_X_train, PCA_y_train)
print metrics.confusion_matrix(PCA_y_train, predictions_train)
print metrics.classification_report(PCA_y_train, predictions_train)
print metrics.accuracy_score(PCA_y_train, predictions_train)

predictions_test = LogReg.predict(PCA_X_test)
score_train = LogReg.score(PCA_X_test, PCA_y_test)
print metrics.confusion_matrix(PCA_y_test, predictions_test)
print metrics.classification_report(PCA_y_test, predictions_test)
print metrics.accuracy_score(PCA_y_test, predictions_test)

[[112  49]
 [ 49 111]]
             precision    recall  f1-score   support

          0       0.70      0.70      0.70       161
          1       0.69      0.69      0.69       160

avg / total       0.69      0.69      0.69       321

0.694704049844
[[33  7]
 [ 9 32]]
             precision    recall  f1-score   support

          0       0.79      0.82      0.80        40
          1       0.82      0.78      0.80        41

avg / total       0.80      0.80      0.80        81

0.802469135802


LDA

In [50]:
# LDA with a random number of components  
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

lda = LinearDiscriminantAnalysis()
lda_data = lda.fit_transform(X_train, y_train)

In [51]:
# Splitting the test and training set from the LDA data
LDA_X_train, LDA_X_test, LDA_y_train, LDA_y_test = train_test_split(lda_data, y_train, test_size=0.2,  random_state = 10) 

In [52]:
# Building a LogReg model from the split of the LDA data

LogReg = LogisticRegression()
LogReg.fit(lda.transform(LDA_X_train), LDA_y_train)

LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
          intercept_scaling=1, max_iter=100, multi_class='ovr', n_jobs=1,
          penalty='l2', random_state=None, solver='liblinear', tol=0.0001,
          verbose=0, warm_start=False)

In [53]:
predictions_train = LogReg.predict(LDA_X_train)
score_train = LogReg.score(LDA_X_train, LDA_y_train)
print metrics.confusion_matrix(LDA_y_train, predictions_train)
print metrics.classification_report(LDA_y_train, predictions_train)
print metrics.accuracy_score(LDA_y_train, predictions_train)

predictions_test = LogReg.predict(LDA_X_test)
score_train = LogReg.score(LDA_X_test, LDA_y_test)
print metrics.confusion_matrix(LDA_y_test, predictions_test)
print metrics.classification_report(LDA_y_test, predictions_test)
print metrics.accuracy_score(LDA_y_test, predictions_test)

[[108  15]
 [ 17 116]]
             precision    recall  f1-score   support

          0       0.86      0.88      0.87       123
          1       0.89      0.87      0.88       133

avg / total       0.88      0.88      0.88       256

0.875
[[33  5]
 [ 3 24]]
             precision    recall  f1-score   support

          0       0.92      0.87      0.89        38
          1       0.83      0.89      0.86        27

avg / total       0.88      0.88      0.88        65

0.876923076923


Regular Data
training accuracy: 100%
test accuracy: 78%

PCA 
training accuracy: 70%
test accuracy: 80%

LDA 
training accuracy: 87.5%
test accuracy: 87.7%


The PCA and LDA results both show they did better than a simple Logistic Regression on my data. Below I outline what the results are for PDA and LDA, and why the differences in their intention provide different results. 

LDA
The purpose of Linear Discriminant Analysis is to combine features of the image to maximize the classification of the images. Specifically it finds the optimal tradeoff between the difference in means of the classes and standard deviation. The balance allows for the datasets to be classified well through a large difference in means (keeping the datasets far apart), balanced with a small standard deviation (datasets don't mix at the edges). In our results, the accuracy score for LDA was highest among all three test runs, and higher than PCA for the training run. This makes sense compared to both because we put in all of the work for LDA to get a better classification than keeping the data normal. This result is supported by the results of PCA – discussed further below – through the general idea that LDA is used specifically for maximizing classification whereas PCA is used for correlation between variables. 

PCA
The results from applying Principle Component Analysis showed a higher accuracy score for the test round than not applying anything to the dataset, but was lower than LDA on training and test (as discussed above). PCA is used primarily to convert your observations into a lower dimension of principal components, with the PC of greatest variance given the greatest weight when measuring correlation. 


Our dataset was very small, consisting of < 200 pictures for each category and the number of components was likely much larger than the sample size (intuitively thinking about how many dimensions there are for the image). I am curious of how the number of components, when compared to the sample size, changes how much we overfit on our training dataset, and reduce our ability to predict for the test data. This seems to matter less for PCA and LDA than it does for running logistic regression on our original dataset, which overfit on the test data and reduced predictive power for the test set. 